# Dynamic Programming

> extension of the theory of Markov Chains to Markov Decision Processes (MDP)

A *MDP* is a stochastic process caractherised by the following quintuplet: $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$, where:

- $\mathcal{S}$ is the set of states;

- $\mathcal{A}$ is the set of actions;

- $\mathcal{P}:=\{P^{a}: a\in\mathcal{A}\}$ is the set of transition matrices for each fixed action;

- $\mathcal{R}$ is the set of rewards obtained by applying an action to a state. We denote by $R_{n+1}$ the reward obtained by applying the action $A_{n}$ to state $S_{n}$, at time $n\in\mathbb{N}^{+}$;

- $\gamma\in [0,1]$ is the penalisation factor.

Instaed of dealing with random variable rewards $R_{n+1}$, we usually consider:

- the average reward $\bar{R}_{n+1}(s,a)=\mathbb{E}[R_{n+1} \mid S_{n}=s, A_{n}=a]$, kwnowing both the state and action;

- the average reward $\bar{R}_{n+1}(s)=\mathbb{E}[R_{n+1} \mid S_{n}=s]$, kwnowing only the state.

We call a *policy* $\pi$ the set of markovian decision rules $\{ \mathbb{P}(A_{n}=a \mid S_{n}=s) : n\in \mathbb{N}^{+}, a\in \mathcal{A}, s\in\mathcal{S}\}$.

In our code, we will only consider *stationary policies*, i.e. not depending on time $n\in\mathbb{N}^{+}$.

**What is the goal of Dynamic Programming ?** 

In a MDP, at each time $n\in \mathbb{N}^{+}$, an action $A_{n}=a$ is applied to a state $S_{n}=s$, and the system receives a reward. Usually, the average value of the reward is positive if we want to encourage this action (good for the system), negative if we want to discourage (bad for the system), or zero if it is ininfluential for the system. The goal of Dynamical Programming is learning which choise of actions maximises the reward.

## Value Function

> the average future reward

Considering $\mathcal{S}$ finite, there are two ways of computing the value function $v_{\pi}$:

1) exactly by inverting a matrix (affordable method if the state space is small);

2) as the limit of a converging sequence.

### 1) Exact method:

 $\quad v_{\pi} = \left(I - \gamma P_{\pi}\right)^{-1} \bar{R}_{\pi}$,

where $I$ is the squared matrix of dimension $|\mathcal{S}|\times |\mathcal{S}|$, $P_{\pi}$ is the transition matrix of the chain of states, and $R_{\pi}$ is the vector of average rewards at each state.


In [ ]:
import numpy as np

! pip install simpy

In [7]:
! pip install simpy


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:

def value_function_exact(gamma, P, R):

    size_S = np.shape(P)[0]

    id = np.eye(size_S)

    inv = np.linalg.inv(id - gamma*P)

    return np.dot(inv, R)
    

Example from course. The particularity here is that we have three absorbing states, for which the rewards are set to zero by convention.

In [ ]:
P_ex = np.array([
    [0.5, 0.5, 0, 0, 0, 0, 0],
    [0.3, 0, 0.7, 0, 0, 0, 0],
    [0, 0, 0.5, 0.5, 0, 0, 0],
    [0, 0, 0, 0.1, 0, 0.7, 0],
    [0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1]
])

R_ex = np.array([0.5, -0.7, -5.5, 89, 0, 0, 0]).T

gamma_ex = 0.8

value_function_exact(gamma_ex, P_ex, R_ex)


### 2) Iterative method:

$\begin{cases}
v_{0} = v_{0}' \\
v_{n+1} = L_{\pi} v_{n}
\end{cases}$,


where $L_{\pi}v(s) = \bar{R}_{\pi}(s) + \gamma \mathbb{E}_{\pi}\left[v(S_{1}) | v(S_{0})=s\right]$, for each $v \in \mathbb{R}^{|\mathcal{S}|}$.

In [ ]:

def value_function_iterative(gamma, P, R, tol=1e-8, max_iter=10000):
    """Evaluate a fixed policy by iterating its Bellman expectation equation."""
    P = np.asarray(P, dtype=float)
    R = np.asarray(R, dtype=float)

    if P.ndim != 2 or P.shape[0] != P.shape[1]:
        raise ValueError("P must be a square transition matrix.")
    if R.shape != (P.shape[0],):
        raise ValueError("R must contain one reward per state.")
    if not 0 <= gamma < 1:
        raise ValueError("gamma must satisfy 0 <= gamma < 1 for convergence.")
    if tol <= 0:
        raise ValueError("tol must be positive.")
    if max_iter <= 0:
        raise ValueError("max_iter must be positive.")

    V = np.zeros_like(R)
    for _ in range(max_iter):
        V_new = R + gamma * (P @ V)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new

    raise RuntimeError("Value iteration did not converge within max_iter.")


V_iterative = value_function_iterative(gamma_ex, P_ex, R_ex)
print("Iterative value function:", V_iterative)
